# Module 1 — YOLOv11-seg: Strawberry Detection & Segmentation
**AIDA 2158A Final Project | Mark Miller**

This notebook covers:
1. Converting val instance maps → YOLO polygon `.txt` labels
2. Writing `data.yaml` for the local dataset
3. Training `yolo11s-seg` on the strawberry dataset
4. Validating and plotting results
5. Running inference to generate ROI crops (Phase 3)

## Cell 1 — Imports & Path Setup

In [1]:
import os
import sys
import glob
import shutil
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image
import torch

# Project root — all paths are relative to this
PROJECT = r"C:\Users\markm\Documents\RDP\AIDA 2158\Final Project"

TRAIN1_IMGS  = os.path.join(PROJECT, "train-20260416T035115Z-3-001", "train", "images")
TRAIN1_LBLS  = os.path.join(PROJECT, "train-20260416T035115Z-3-001", "train", "labels")
TRAIN2_IMGS  = os.path.join(PROJECT, "train-20260416T035115Z-3-002", "train", "images")
TRAIN2_LBLS  = os.path.join(PROJECT, "train-20260416T035115Z-3-002", "train", "labels")
VAL_IMGS     = os.path.join(PROJECT, "val", "val", "images")
VAL_NB       = os.path.join(PROJECT, "val", "val", "labels_non_binary")
VAL_LBLS     = os.path.join(PROJECT, "val", "val", "labels")      # will be created
TEST_IMGS    = os.path.join(PROJECT, "test-20260416T035806Z-3-001", "test", "images")
TEST_LBLS    = os.path.join(PROJECT, "test-20260416T035806Z-3-001", "test", "labels")
YAML_PATH    = os.path.join(PROJECT, "strawberry_seg.yaml")
ROI_DIR      = os.path.join(PROJECT, "roi_crops")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Quick dataset count check
for name, folder in [("Train-1 images", TRAIN1_IMGS), ("Train-1 labels", TRAIN1_LBLS),
                     ("Train-2 images", TRAIN2_IMGS), ("Train-2 labels", TRAIN2_LBLS),
                     ("Val images",     VAL_IMGS),    ("Val NB labels",  VAL_NB),
                     ("Test images",    TEST_IMGS),   ("Test labels",    TEST_LBLS)]:
    n = len(os.listdir(folder)) if os.path.exists(folder) else "MISSING"
    print(f"  {name}: {n}")

PyTorch: 2.10.0.dev20251205+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Laptop GPU
  Train-1 images: 1496
  Train-1 labels: 1389
  Train-2 images: 1304
  Train-2 labels: 1411
  Val images: 100
  Val NB labels: 100
  Test images: 200
  Test labels: 200


## Cell 2 — Convert Val Instance Maps → YOLO Polygon Labels

The val set has `labels_non_binary/` PNG files where each pixel value = a strawberry instance ID.
YOLO needs `.txt` files with one line per instance: `class x1 y1 x2 y2 ... xn yn` (normalised 0–1).

We extract the contour of each instance and write it as a polygon.

In [2]:
os.makedirs(VAL_LBLS, exist_ok=True)

nb_files = [f for f in os.listdir(VAL_NB) if f.lower().endswith('.png')]
converted = 0
skipped   = 0

for fname in nb_files:
    stem   = os.path.splitext(fname)[0]
    nb_img = np.array(Image.open(os.path.join(VAL_NB, fname)))
    h, w   = nb_img.shape[:2]
    
    lines = []
    for inst_id in np.unique(nb_img):
        if inst_id == 0:
            continue   # background
        mask = (nb_img == inst_id).astype(np.uint8) * 255
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
        # Use the largest contour for this instance
        contour = max(contours, key=cv2.contourArea)
        if cv2.contourArea(contour) < 50:   # skip tiny artefacts
            continue
        # Simplify to reduce point count while preserving shape
        epsilon = 0.002 * cv2.arcLength(contour, True)
        approx  = cv2.approxPolyDP(contour, epsilon, True)
        coords  = approx.reshape(-1, 2)
        if len(coords) < 3:
            continue
        # Normalise to 0-1
        norm = []
        for x, y in coords:
            norm.extend([x / w, y / h])
        line = "0 " + " ".join(f"{v:.6f}" for v in norm)
        lines.append(line)
    
    out_path = os.path.join(VAL_LBLS, stem + ".txt")
    if lines:
        with open(out_path, 'w') as f:
            f.write("\n".join(lines))
        converted += 1
    else:
        # Write empty file so YOLO doesn't complain about missing label
        open(out_path, 'w').close()
        skipped += 1

print(f"Val labels converted: {converted}  |  Empty (no instances): {skipped}")
print(f"Total .txt files in {VAL_LBLS}: {len(os.listdir(VAL_LBLS))}")

Val labels converted: 100  |  Empty (no instances): 0
Total .txt files in C:\Users\markm\Documents\RDP\AIDA 2158\Final Project\val\val\labels: 100


## Cell 3 — Write data.yaml

In [3]:
# YOLO needs forward slashes in paths
def to_fwd(p):
    return p.replace("\\", "/")

yaml_content = f"""# Strawberry segmentation dataset — AIDA 2158A Final Project
path: {to_fwd(PROJECT)}

train:
  - {to_fwd(TRAIN1_IMGS)}
  - {to_fwd(TRAIN2_IMGS)}

val:  {to_fwd(VAL_IMGS)}
test: {to_fwd(TEST_IMGS)}

nc: 1
names:
  0: Strawberry
"""

with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

print("Written:", YAML_PATH)
print()
print(yaml_content)

Written: C:\Users\markm\Documents\RDP\AIDA 2158\Final Project\strawberry_seg.yaml

# Strawberry segmentation dataset — AIDA 2158A Final Project
path: C:/Users/markm/Documents/RDP/AIDA 2158/Final Project

train:
  - C:/Users/markm/Documents/RDP/AIDA 2158/Final Project/train-20260416T035115Z-3-001/train/images
  - C:/Users/markm/Documents/RDP/AIDA 2158/Final Project/train-20260416T035115Z-3-002/train/images

val:  C:/Users/markm/Documents/RDP/AIDA 2158/Final Project/val/val/images
test: C:/Users/markm/Documents/RDP/AIDA 2158/Final Project/test-20260416T035806Z-3-001/test/images

nc: 1
names:
  0: Strawberry



## Cell 4 — Train YOLOv11-seg

**Key hyperparameters:**
| Parameter | Value | Reason |
|---|---|---|
| model | yolo11s-seg | Recommended by project spec; small = faster to train |
| epochs | 50 | Enough to converge; can extend if curves still improving |
| imgsz | 640 | Standard YOLO input size; good balance of speed vs accuracy |
| batch | 8 | Fits comfortably in RTX 5070 VRAM |
| optimizer | AdamW | Default for YOLO11; better than SGD for small datasets |
| patience | 15 | Early stopping if val mAP stops improving |

Training will save results to `runs/segment/strawberry_seg/`.

In [4]:
from ultralytics import YOLO

model = YOLO("yolo11s-seg.pt")   # downloads pretrained weights on first run (~20 MB)

results = model.train(
    data      = YAML_PATH,
    epochs    = 50,
    imgsz     = 640,
    batch     = 8,
    optimizer = "AdamW",
    patience  = 15,
    device    = 0,              # GPU 0 (RTX 5070)
    project   = os.path.join(PROJECT, "runs"),
    name      = "strawberry_seg",
    exist_ok  = True,
    verbose   = True,
)

New https://pypi.org/project/ultralytics/8.4.38 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.235  Python-3.11.14 torch-2.10.0.dev20251205+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\markm\Documents\RDP\AIDA 2158\Final Project\strawberry_seg.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-seg.pt, momentum=0

## Cell 5 — Validate & Print Metrics

In [5]:
# Load best weights and validate
best_weights = os.path.join(PROJECT, "runs", "strawberry_seg", "weights", "best.pt")
model_best   = YOLO(best_weights)

metrics = model_best.val(
    data   = YAML_PATH,
    imgsz  = 640,
    device = 0,
)

print("\n=== Validation Results ===")
print(f"  mAP50 (box):    {metrics.box.map50:.4f}")
print(f"  mAP50-95 (box): {metrics.box.map:.4f}")
print(f"  mAP50 (seg):    {metrics.seg.map50:.4f}")
print(f"  mAP50-95 (seg): {metrics.seg.map:.4f}")
print(f"  Precision:      {metrics.box.mp:.4f}")
print(f"  Recall:         {metrics.box.mr:.4f}")

Ultralytics 8.3.235  Python-3.11.14 torch-2.10.0.dev20251205+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Laptop GPU, 8151MiB)
YOLO11s-seg summary (fused): 113 layers, 10,067,203 parameters, 0 gradients, 32.8 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 2232.5108.3 MB/s, size: 1454.0 KB)
val: Scanning C:\Users\markm\Documents\RDP\AIDA 2158\Final Project\val\val\labels.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 50.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.9it/s 2.4s0.3ss
                   all        100        570      0.903      0.846      0.927      0.768       0.91      0.846       0.92      0.712
Speed: 1.3ms preprocess, 8.4ms inference, 0.0ms loss, 3.8ms postprocess per image
Results saved to C:\Users\markm\Documents\RDP\AIDA 2158\Final Project\runs\segment\val

=== Validation Results ===
  mAP50 (box):    0.9265
  mAP50-9

## Cell 6 — Plot Training Curves

YOLO saves a `results.csv` with per-epoch metrics. We plot:
- Training vs validation loss
- Segmentation mAP over epochs

In [6]:
import pandas as pd

results_csv = os.path.join(PROJECT, "runs", "strawberry_seg", "results.csv")
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()   # remove accidental whitespace

print("Columns available:", list(df.columns))
print(df.tail(5))

Columns available: ['epoch', 'time', 'train/box_loss', 'train/seg_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/precision(M)', 'metrics/recall(M)', 'metrics/mAP50(M)', 'metrics/mAP50-95(M)', 'val/box_loss', 'val/seg_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']
    epoch     time  train/box_loss  train/seg_loss  train/cls_loss  \
43     44  2546.75         0.59079         0.77555         2.36296   
44     45  2602.15         0.58201         0.76493         2.20189   
45     46  2657.80         0.55135         0.74441         2.45712   
46     47  2714.19         0.57470         0.75612         2.15425   
47     48  2769.76         0.55872         0.71918         2.32644   

    train/dfl_loss  metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  \
43         0.86924               0.87791            0.82000           0.90641   
44         0.87278               0.887

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("YOLOv11-seg Training — Strawberry Dataset", fontsize=14, fontweight='bold')

epochs = df["epoch"]

# --- Loss curves ---
ax = axes[0]
# Box loss
if "train/box_loss" in df.columns:
    ax.plot(epochs, df["train/box_loss"],      label="Train box loss")
    ax.plot(epochs, df["val/box_loss"],        label="Val box loss", linestyle="--")
# Seg loss
if "train/seg_loss" in df.columns:
    ax.plot(epochs, df["train/seg_loss"],      label="Train seg loss")
    ax.plot(epochs, df["val/seg_loss"],        label="Val seg loss",  linestyle="--")
ax.set_title("Loss Curves")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
ax.grid(True, alpha=0.3)

# --- mAP (box) ---
ax = axes[1]
if "metrics/mAP50(B)" in df.columns:
    ax.plot(epochs, df["metrics/mAP50(B)"],    label="mAP50 (box)")
    ax.plot(epochs, df["metrics/mAP50-95(B)"], label="mAP50-95 (box)", linestyle="--")
ax.set_title("Box mAP")
ax.set_xlabel("Epoch")
ax.set_ylabel("mAP")
ax.legend()
ax.grid(True, alpha=0.3)

# --- mAP (seg) ---
ax = axes[2]
if "metrics/mAP50(M)" in df.columns:
    ax.plot(epochs, df["metrics/mAP50(M)"],    label="mAP50 (seg)")
    ax.plot(epochs, df["metrics/mAP50-95(M)"], label="mAP50-95 (seg)", linestyle="--")
ax.set_title("Segmentation mAP")
ax.set_xlabel("Epoch")
ax.set_ylabel("mAP")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = os.path.join(PROJECT, "runs", "strawberry_seg", "training_curves.png")
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {plot_path}")

<Figure size 1800x500 with 3 Axes>

Saved: C:\Users\markm\Documents\RDP\AIDA 2158\Final Project\runs\strawberry_seg\training_curves.png


## Cell 7 — Hyperparameter Table (Submission)

In [8]:
import pandas as pd

hyperparams = {
    "Parameter":   ["Model",         "Epochs", "Image size", "Batch size", "Optimizer", "Learning rate", "Early stopping patience", "Device"],
    "Value":       ["yolo11s-seg.pt", 50,        640,          8,            "AdamW",     "auto (0.01)",   15,                        "RTX 5070 (CUDA)"],
    "Reason":      [
        "Project-recommended small segmentation model",
        "Sufficient for convergence on ~2800 training images",
        "Standard YOLO resolution; balances speed and accuracy",
        "Fits RTX 5070 VRAM; stable gradient estimation",
        "Better convergence than SGD for segmentation tasks",
        "YOLO default cosine-annealed schedule",
        "Stops training if val mAP stops improving",
        "NVIDIA GeForce RTX 5070 Laptop GPU",
    ]
}

hp_df = pd.DataFrame(hyperparams)
print(hp_df.to_string(index=False))

              Parameter           Value                                                Reason
                  Model  yolo11s-seg.pt          Project-recommended small segmentation model
                 Epochs              50   Sufficient for convergence on ~2800 training images
             Image size             640 Standard YOLO resolution; balances speed and accuracy
             Batch size               8        Fits RTX 5070 VRAM; stable gradient estimation
              Optimizer           AdamW    Better convergence than SGD for segmentation tasks
          Learning rate     auto (0.01)                 YOLO default cosine-annealed schedule
Early stopping patience              15             Stops training if val mAP stops improving
                 Device RTX 5070 (CUDA)                    NVIDIA GeForce RTX 5070 Laptop GPU


## Cell 8 — Sample Predictions on Val Set
Visual check: overlay predicted masks on 6 val images.

In [9]:
val_images = sorted(glob.glob(os.path.join(VAL_IMGS, "*.png")))[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("YOLOv11-seg Predictions — Val Samples", fontsize=14, fontweight='bold')

for ax, img_path in zip(axes.flat, val_images):
    result = model_best.predict(img_path, imgsz=640, device=0, verbose=False)[0]
    # result.plot() returns a BGR numpy array with masks overlaid
    annotated = result.plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated_rgb)
    n_det = len(result.boxes) if result.boxes is not None else 0
    ax.set_title(f"{os.path.basename(img_path)} — {n_det} detected", fontsize=9)
    ax.axis('off')

plt.tight_layout()
pred_path = os.path.join(PROJECT, "runs", "strawberry_seg", "val_predictions_sample.png")
plt.savefig(pred_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {pred_path}")

<Figure size 1800x1000 with 6 Axes>

Saved: C:\Users\markm\Documents\RDP\AIDA 2158\Final Project\runs\strawberry_seg\val_predictions_sample.png


## Cell 9 — Generate ROI Crops (Phase 3)

For every image in the dataset:
1. Run YOLOv11 inference
2. Find the largest detected strawberry (by mask area)
3. Crop that region with a 20px pad
4. Save to `roi_crops/`

This is the input to Phase 4 (peduncle annotation) and Phase 5 (U-Net training).

In [12]:
os.makedirs(ROI_DIR, exist_ok=True)

# Gather all images from all splits
all_img_paths = (
    sorted(glob.glob(os.path.join(TRAIN1_IMGS, "*.png"))) +
    sorted(glob.glob(os.path.join(TRAIN2_IMGS, "*.png"))) +
    sorted(glob.glob(os.path.join(VAL_IMGS,    "*.png"))) +
    sorted(glob.glob(os.path.join(TEST_IMGS,   "*.png")))
)
print(f"Total images to process: {len(all_img_paths)}")

PAD          = 20     # pixels to pad around the detected strawberry
saved        = 0
no_detection = 0

for img_path in all_img_paths:
    stem   = os.path.splitext(os.path.basename(img_path))[0]
    out_p  = os.path.join(ROI_DIR, stem + ".png")
    if os.path.exists(out_p):          # skip already done
        saved += 1
        continue

    result = model_best.predict(img_path, imgsz=640, device=0, conf=0.01, verbose=False)[0]

    if result.boxes is None or len(result.boxes) == 0:
        no_detection += 1
        continue

    img = cv2.imread(img_path)
    H, W = img.shape[:2]

    # Find largest strawberry by bounding box area
    boxes = result.boxes.xyxy.cpu().numpy()   # [N, 4] x1 y1 x2 y2
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    best  = int(np.argmax(areas))
    x1, y1, x2, y2 = boxes[best].astype(int)

    # Pad and clamp to image bounds
    x1 = max(0, x1 - PAD)
    y1 = max(0, y1 - PAD)
    x2 = min(W, x2 + PAD)
    y2 = min(H, y2 + PAD)

    crop = img[y1:y2, x1:x2]
    cv2.imwrite(out_p, crop)
    saved += 1

    if saved % 200 == 0:
        print(f"  {saved} crops saved...")

print(f"\nDone.")
print(f"  ROI crops saved: {saved}")
print(f"  Images with no detection: {no_detection}")
print(f"  Crops folder: {ROI_DIR}")

Total images to process: 3100
  200 crops saved...
  400 crops saved...
  600 crops saved...
  800 crops saved...
  1000 crops saved...
  1200 crops saved...
  1400 crops saved...
  1600 crops saved...
  2000 crops saved...
  2200 crops saved...
  2400 crops saved...
  2600 crops saved...
  2800 crops saved...
  3000 crops saved...

Done.
  ROI crops saved: 3100
  Images with no detection: 0
  Crops folder: C:\Users\markm\Documents\RDP\AIDA 2158\Final Project\roi_crops


## Cell 10 — Visualise ROI Crops
Show a grid of 9 sample crops to verify they look correct.

In [13]:
crop_files = sorted(glob.glob(os.path.join(ROI_DIR, "*.png")))[:9]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
fig.suptitle("ROI Crops — Largest Strawberry per Image", fontsize=14, fontweight='bold')

for ax, path in zip(axes.flat, crop_files):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(os.path.basename(path), fontsize=8)
    ax.axis('off')

# Hide unused subplots
for ax in axes.flat[len(crop_files):]:
    ax.axis('off')

plt.tight_layout()
roi_grid_path = os.path.join(PROJECT, "runs", "strawberry_seg", "roi_crops_sample.png")
plt.savefig(roi_grid_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {roi_grid_path}")

<Figure size 1500x1200 with 9 Axes>

Saved: C:\Users\markm\Documents\RDP\AIDA 2158\Final Project\runs\strawberry_seg\roi_crops_sample.png
